In [ ]:
import os
import json
import yaml
import pandas as pd
from glob import glob
from ultralytics import YOLO
from IPython.display import display

In [2]:
SELECTED_MODEL = 'yolov8m'

WS_PATH = os.path.dirname(os.getcwd())
SRC_PATH = os.path.join(WS_PATH, 'src')
IMAGE_DIR = os.path.join(SRC_PATH, 'extracted_frames')
RESULT_PATH = os.path.join(SRC_PATH, 'training_results')
MODEL_PATH = os.path.join(RESULT_PATH, SELECTED_MODEL, 'weights', 'best.pt')

CLASS_NAMES = [
    'A',
    'B',
    'C',
    'D',
    'E',
    'F',
    'G',
    'H',
    'I',
    'J',
    'K',
    'L',
    'M',
    'N',
    'O',
    'P',
    'Q',
    'R',
    'S',
    'T',
    'U',
    'V',
    'W',
    'X',
    'Y',
    'Z',
    'apa',
    'ayah',
    'cinta',
    'halo',
    'hati_hati',
    'ibu',
    'jangan',
    'kakak',
    'kakek',
    'kamu',
    'kerja',
    'maaf',
    'makan',
    'marah',
    'minum',
    'mobil',
    'motor',
    'nenek',
    'rumah',
    'sahabat',
    'sakit',
    'saya',
    'sedih',
    'senang',
    'siapa',
    'takut',
    'terima_kasih',
    'tidur',
    'tolong',
    'uang',
    'background'
]

BACKGROUND_IDX = len(CLASS_NAMES)

In [15]:
# Configuration
BACKGROUND_IMAGE_DIR = os.path.join(SRC_PATH, 'background_images')

# Load the model
model = YOLO(MODEL_PATH)

def convert_labelme_to_yolo(json_path, txt_path, class_names):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    img_w = data['imageWidth']
    img_h = data['imageHeight']
    
    yolo_lines = []
    for shape in data.get('shapes', []):
        label = shape['label'].replace(" ", "_")
        if label not in class_names:
            continue
            
        class_id = class_names.index(label)
        
        points = shape['points']
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        
        x_center = ((x_min + x_max) / 2) / img_w
        y_center = ((y_min + y_max) / 2) / img_h
        width = (x_max - x_min) / img_w
        height = (y_max - y_min) / img_h
        
        # Ensure values are within 0 and 1
        x_center = max(0.0, min(1.0, x_center))
        y_center = max(0.0, min(1.0, y_center))
        width = max(0.0, min(1.0, width))
        height = max(0.0, min(1.0, height))
        
        yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
        
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(yolo_lines))

results = []
backgrounds = [d for d in os.listdir(BACKGROUND_IMAGE_DIR) if os.path.isdir(os.path.join(BACKGROUND_IMAGE_DIR, d))]

for bg in backgrounds:
    print(f"\n{'='*40}")
    print(f"Evaluating background: {bg}")
    print(f"{'='*40}")
    
    bg_dir = os.path.join(BACKGROUND_IMAGE_DIR, bg)
    json_files = glob(os.path.join(bg_dir, '*.json'))
    
    image_paths = []
    
    for j_path in json_files:
        base_name = os.path.splitext(j_path)[0]
        txt_path = base_name + '.txt'
        
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png']:
            if os.path.exists(base_name + ext):
                img_path = base_name + ext
                break
                
        if img_path:
            convert_labelme_to_yolo(j_path, txt_path, CLASS_NAMES)
            image_paths.append(os.path.abspath(img_path))
            
    if not image_paths:
        print(f"No images found for {bg}. Skipping.")
        continue
        
    # YOLO validation txt and yaml
    train_txt_path = os.path.join(bg_dir, 'train_images.txt')
    val_txt_path = os.path.join(bg_dir, 'val_images.txt')
    with open(val_txt_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(image_paths))
        
    yaml_path = os.path.join(bg_dir, f'{bg}_dataset.yaml')
    yaml_data = {
        'path': os.path.abspath(bg_dir),
        'train': os.path.abspath(train_txt_path),
        'val': os.path.abspath(val_txt_path),
        'names': {i: name for i, name in enumerate(CLASS_NAMES)}
    }
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(yaml_data, f)
        
    # Validate using ultralytics
    # YOLO val automatically searches for labels by replacing images with labels or looks in same directory
    metrics = model.val(data=yaml_path, split='val', plots=False, verbose=False)
    
    p = metrics.box.mp
    r = metrics.box.mr
    map50 = metrics.box.map50
    map50_95 = metrics.box.map
    l = metrics.speed['inference']
    
    print(f"Precision: {p:.4f}")
    print(f"Recall: {r:.4f}")
    print(f"mAP@50: {map50:.4f}")
    print(f"mAP@50-95: {map50_95:.4f}")
    print(f"Latency: {l:.4f}")
    
    results.append({
        'Background': bg,
        'Images': len(image_paths),
        'Precision': p,
        'Recall': r,
        'mAP@50': map50,
        'mAP@50-95': map50_95,
        'Latency': l
    })

df_results = pd.DataFrame(results)
print("\n\n========================================")
print("FINAL SUMMARY: mAP PER BACKGROUND")
print("========================================")
display(df_results)



Evaluating background: close-up
Ultralytics 8.4.52  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1050, 3072MiB)
Model summary (fused): 93 layers, 25,872,184 parameters, 0 gradients, 78.9 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 127.225.6 MB/s, size: 40.3 KB)
val: Scanning e:\Kuliah\Skripsi\Project\BISINDO-Object-Detection-Training\src\background_images\close-up.cache... 404 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 404/404  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 1.0s/it 26.2s1.0ss
                   all        404        404      0.974      0.976       0.99       0.91
Speed: 3.3ms preprocess, 53.7ms inference, 0.0ms loss, 2.6ms postprocess per image
Precision: 0.9737
Recall: 0.9764
mAP@50: 0.9896
mAP@50-95: 0.9097
Latency: 53.7341

Evaluating background: lapangan
Ultralytics 8.4.52  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1050, 3072MiB)
val: Fast ima

,Background,Images,Precision,Recall,mAP@50,mAP@50-95,Latency
0,close-up,404,0.973701,0.976380,0.989596,0.909664,53.734089
1,lapangan,404,0.974675,0.988042,0.992890,0.940921,53.944054
2,lorong,404,0.978554,0.987738,0.990763,0.917320,53.878596
3,taman,404,0.973955,0.979760,0.979659,0.833015,53.869939
4,tembok,404,0.978652,0.992024,0.992727,0.924741,53.905947
